# DPR processor example with Prefect+Dask

In [1]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['PREFECT_PUBLIC']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

Internal Prefect server: http://prefect-server:4200/api
Public Prefect dashboard: http://localhost:4200/dashboard


In [2]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
init_demo()
init_dask_cluster_eopf(scale=4)

# In local mode, init the prefect blocks.
# NOTE: In the cluster, the blocks must be created only once by the admin.
await init_prefect_blocks()

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

Auxip service: http://rs-server-adgs:8000
CADIP service: http://rs-server-cadip:8000
Catalog service: http://rs-server-catalog:8000
Connecting to dask gateway for 'dask-eopf': http://dask-eopf:8000 ...
Dask dashboard for 'dask-eopf': http://localhost:8702/clusters/12b8299482f64173bdbd3b3a1fbaf33d/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


In [3]:
%%bash
prefect block ls
echo -e "\nNOTE: you can see the block details and credentials by running e.g.: prefect block inspect remote-file-system/s3"

                                     Blocks                                     
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ ID                                ┃ Type            ┃ … ┃ Slug               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ 1a97681e-27d4-4b88-9ff4-447ac19a… │ Remote File Sy… │ … │ remote-file-syste… │
└───────────────────────────────────┴─────────────────┴───┴────────────────────┘
                 List Block Types using `prefect block type ls`                 

NOTE: you can see the block details and credentials by running e.g.: prefect block inspect remote-file-system/s3


In [4]:
%%bash
mkdir -p /tmp/myzarr
touch /tmp/myzarr/empty.file

In [5]:
# Upload local directory contents
await PREFECT_BLOCK_S3.put_directory(local_path = "/tmp/myzarr", to_path = "myzarr")

1

In [6]:
import os

gateway = dask_gateway_eopf
client = dask_client_eopf
cluster = dask_cluster_eopf

os.environ["DASK_GATEWAY_ADDRESS"] = gateway.address
os.environ["DASK_CLUSTER_NAME"] = cluster.name

from dpr_processor_example import dpr_flow
dpr_flow("/tmp/eopf_result")


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


17:05:15.590 | INFO    | prefect.engine - Created flow run 'magenta-quokka' for flow 'dpr-flow'

17:05:15.593 | INFO    | prefect.engine - View at http://prefect-server:4200/runs/flow-run/64318c9a-c69f-400e-9345-f3b15f83c381

17:05:15.621 | INFO    | prefect.task_runner.dask - Connecting to existing Dask cluster GatewayCluster<12b8299482f64173bdbd3b3a1fbaf33d, status=running>

/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


17:05:15.641 | WARNING | Flow run 'magenta-quokka' -  IP address for flow: 172.18.0.20

17:05:24.879 | INFO    | Flow run 'magenta-quokka' - Finished in state Completed()

In [ ]:
# https://help.marine.copernicus.eu/en/articles/8077952-how-to-open-and-visualize-zarr-format-data

## 3. Shutdown the dask clusters

In [7]:
# # You can scale the clusters to 0 workers
# dask_gateway_eopf.scale_cluster(dask_cluster_eopf.name, 0)

# # Or shutdown the clusters
# shutdown_dask_clusters(dask_gateway_eopf)

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.